# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, overview, and explore the FAIR^2 dataset with the `mlcroissant` library by referencing all dataset entities via their `@id` fields.

### Dataset Source
The dataset is described by a Croissant schema at:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print high-level description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We enumerate the record sets and fields using their `@id`. This ensures exact references for data extraction and manipulation with `mlcroissant`.


In [ ]:
# List record sets by @id
record_sets = []
if hasattr(metadata, 'record_set') and metadata.record_set:
    record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in metadata.record_set]

if not record_sets:
    # Try dataset._record_sets for datasets where record_set is not populated
    record_sets = [r['@id'] for r in dataset._record_sets.values()]
    print('Record sets found via dataset._record_sets:')
else:
    print('Record sets found via metadata.record_set:')

for rs_id in record_sets:
    print(f"RecordSet @id: {rs_id}")

# For each record set, list fields (via @id)
print("\nFields for each RecordSet:")
for rs_id in record_sets:
    rs_obj = dataset._record_sets[rs_id]
    if 'field' in rs_obj:
        field_ids = [f['@id'] if isinstance(f, dict) and '@id' in f else f for f in rs_obj['field']]
        print(f"\nRecordSet {rs_id}: ")
        for f_id in field_ids:
            print(f"  Field @id: {f_id}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use record set and field `@id`s as shown above.

In [ ]:
# Extract all data tables, using @id for record sets
dataframes = dict()
print('Extracting data for each record set by @id:')

for rs_id in record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"- Loaded RecordSet @id: {rs_id} with shape {df.shape}")
        else:
            print(f"- Loaded RecordSet @id: {rs_id} (no records)")
    except Exception as e:
        print(f"- Failed to load records for RecordSet @id: {rs_id}: {e}")

# List the columns of each DataFrame
for rs_id, df in dataframes.items():
    print(f"\nColumns for RecordSet @id '{rs_id}':\n", df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing: filter rows, normalize numeric fields, and group data by attributes. All operations reference fields by their `@id`.

In [ ]:
# Select a record set for analysis
if dataframes:
    # Use first available record set as example
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using RecordSet @id: {record_set_id}")
    
    # Display available numeric fields by selecting those with numeric dtype
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        print("No numeric fields detected. Attempting to infer numeric fields:")
        possible_numeric = []
        for col in df.columns:
            # Try to coerce to numeric type, ignore errors
            series = pd.to_numeric(df[col], errors='coerce')
            if series.notnull().sum() > 0:
                possible_numeric.append(col)
        numeric_fields = possible_numeric
        print(numeric_fields)
    
    if numeric_fields:
        # Take the first numeric field
        numeric_field_id = numeric_fields[0]
        print(f"Numeric field chosen for EDA: {numeric_field_id}")
        
        # Show some value statistics
        print("\nSummary statistics:")
        print(df[numeric_field_id].describe())

        # Filter: show records with value above certain threshold (use median as example)
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records where {numeric_field_id} > {threshold}:")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field if present
        category_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
        group_field = category_fields[0] if category_fields else None

        if group_field:
            print(f"\nGrouping by categorical field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("\nNo suitable categorical field available for grouping.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No dataframes loaded for EDA.")

## 5. Visualization
Visualize the distribution of the key numeric field or relationships with a categorical attribute.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field_id' in locals() and 'filtered_df' in locals():
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id], bins=10, kde=True, color='royalblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=30, ha='right')
        plt.show()
else:
    print("No numeric field and filtered data available for visualization.")

## 6. Conclusion
This notebook illustrated how to load, overview, and analyze the FAIR^2 dataset using the `mlcroissant` library, referencing record sets and fields by their `@id` throughout, in compliance with best practices for consistent and reproducible data workflows. You can explore the specific details of entities (such as clinical attributes, MSI-H phenotypes, anatomical groups, etc.) by referring to their IDs and schema documentation. Continue to adapt and extend this notebook for custom analyses as needed.